In [ ]:
!nvidia-smi

Sat Jun 20 10:55:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%capture
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install unsloth

In [ ]:
import torch
import unsloth

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

print("Unsloth imported successfully")

PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB
Unsloth imported successfully


In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "oieieio/Finance-Instruct-500k",
    split="train"
)

print(dataset)

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 518185
})


In [ ]:
finance_keywords = [
    # Investing & Markets
    "stock", "stocks", "share", "shares", "equity",
    "bond", "bonds", "portfolio", "investment",
    "investing", "investor", "asset", "assets",
    "security", "securities", "dividend", "yield",
    "capital gain", "capital loss", "return",
    "risk", "volatility", "market", "trading",
    "trader", "option", "options", "futures",
    "derivative", "etf", "mutual fund",

    # Banking & Credit
    "bank", "banking", "loan", "loans",
    "credit", "debt", "mortgage",
    "interest", "interest rate",
    "deposit", "withdrawal",
    "liquidity", "lending",

    # Economics
    "economics", "economy", "economic",
    "inflation", "deflation",
    "recession", "gdp", "fiscal",
    "monetary", "central bank",
    "federal reserve", "money supply",
    "exchange rate",

    # Accounting & Corporate Finance
    "revenue", "profit", "loss",
    "expense", "cost", "margin",
    "earnings", "ebitda",
    "balance sheet",
    "income statement",
    "cash flow",
    "financial statement",
    "valuation",
    "market capitalization",

    # Personal Finance
    "budget", "saving", "savings",
    "retirement", "insurance",
    "tax", "taxation", "wealth",
    "net worth", "financial planning",

    # Crypto
    "crypto", "cryptocurrency",
    "bitcoin", "ethereum",
    "blockchain", "token",
    "wallet"
]


def is_finance_related(example):
    text = (
        example["user"] + " " +
        example["assistant"]
    ).lower()

    return any(
        keyword in text
        for keyword in finance_keywords
    )

In [ ]:
dataset = dataset.filter(
    is_finance_related,
    num_proc=2
)

print(dataset)

Filter (num_proc=2):   0%|          | 0/253739 [00:00<?, ? examples/s]

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 253739
})


In [ ]:
strong_finance_keywords = [
    "stock", "bond", "equity", "portfolio",
    "investment", "investor", "trading",
    "dividend", "asset", "liability",
    "financial", "finance", "bank",
    "loan", "credit", "debt",
    "interest rate", "inflation",
    "recession", "gdp", "fiscal",
    "monetary", "central bank",
    "revenue", "profit", "earnings",
    "cash flow", "balance sheet",
    "income statement", "valuation",
    "market capitalization", "tax",
    "retirement", "insurance",
    "wealth", "budget", "savings",
    "mutual fund", "etf",
    "economy", "economic"
]


negative_keywords = [
    "python", "java", "javascript",
    "c++", "c#", "function",
    "class", "algorithm",
    "code", "program",
    "api", "proto",
    "github", "database",
    "sql", "html", "css",
    "json", "xml",
    "biological", "protein",
    "chemical", "physics",
    "history", "date of death"
]


def clean_finance_filter(example):
    text = (
        example["user"] + " " +
        example["assistant"]
    ).lower()

    has_finance = any(
        keyword in text
        for keyword in strong_finance_keywords
    )

    has_noise = any(
        keyword in text
        for keyword in negative_keywords
    )

    return has_finance and not has_noise

In [ ]:
dataset = dataset.filter(
    clean_finance_filter,
    num_proc=2
)

print(dataset)

Filter (num_proc=2):   0%|          | 0/253739 [00:00<?, ? examples/s]

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 157298
})


In [ ]:
dataset = dataset.shuffle(seed=3407)

for i in range(5):
    print(f"\n=== Example {i+1} ===")
    print("USER:", dataset[i]["user"][:300])
    print("\nASSISTANT:", dataset[i]["assistant"][:300])
    print("-" * 80)


=== Example 1 ===
USER: Read this headline: "Gold futures down at Rs 27,835 on weak global cues"
Now answer this question: "Does the news headline talk about price in the past?"
Options:
- No
- Yes Yes

Read this headline: "Acacia Mining lifts FY gold production guidance"
Now answer this question: "Does the news headline t

ASSISTANT: No
--------------------------------------------------------------------------------

=== Example 2 ===
USER: Alex needs to borrow $\\$10,\\!000$ from the bank. The bank gives him two options.

1. A ten-year loan with an annual interest rate of $10\\%$ compounded quarterly, with the condition that at the end of 5 years, Alex must make a payment equal to half of what he owes.  The other half continues to acc

ASSISTANT: We will calculate the total amounts Alex has to pay for each option and then find the positive difference.

Option 1:

Given: $P = \\$10,\\!000$, $r = 10\\% = 0.1$ per year, compounded quarterly, so $n = 4$, and time $t = 5$ years for the f

In [ ]:
def formatting_prompts_func(examples):
    conversations = []

    for user, assistant in zip(
        examples["user"],
        examples["assistant"]
    ):
        messages = [
            {
                "role": "system",
                "content": (
                    "You are Investo Bot, an expert AI financial assistant. "
                    "Provide clear, accurate, and well-structured financial explanations. "
                    "When discussing investments, explain potential risks and avoid making unrealistic guarantees."
                )
            },
            {
                "role": "user",
                "content": user
            },
            {
                "role": "assistant",
                "content": assistant
            }
        ]

        conversations.append(messages)

    texts = tokenizer.apply_chat_template(
        conversations,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": texts
    }


dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    batch_size=1000,
    num_proc=2
)

Map (num_proc=2):   0%|          | 0/157298 [00:00<?, ? examples/s]

In [ ]:
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 20 Jun 2026

You are Investo Bot, an expert AI financial assistant. Provide clear, accurate, and well-structured financial explanations. When discussing investments, explain potential risks and avoid making unrealistic guarantees.<|eot_id|><|start_header_id|>user<|end_header_id|>

Read this headline: "Gold futures down at Rs 27,835 on weak global cues"
Now answer this question: "Does the news headline talk about price in the past?"
Options:
- No
- Yes Yes

Read this headline: "Acacia Mining lifts FY gold production guidance"
Now answer this question: "Does the news headline talk about price staying constant?"
Options:
- No
- Yes No

Read this headline: "Festive demand lifts gold, silver prices"
Now answer this question: "Does the news headline talk about price going down?"
Options:
- No
- Yes No

Read this headline: "gold prices slip lower as dollar remains supported"
Now answ

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,

    args=TrainingArguments(
        # GPU memory control
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # Training length
        max_steps=1000,

        # Optimization
        learning_rate=2e-4,
        warmup_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",

        # Precision
        fp16=True,

        # Logging & saving
        logging_steps=10,
        save_strategy="no",
        output_dir="investo_outputs",

        # Reproducibility
        seed=3407,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/157298 [00:00<?, ? examples/s]

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 157,298 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,0.950936
20,1.002311
30,1.014690
40,1.024197
50,0.988884
60,0.987098
70,0.951379
80,0.898333
90,1.005365
100,0.968127


In [ ]:
model.save_pretrained("investo_lora")
tokenizer.save_pretrained("investo_lora")

Unsloth: Restored added_tokens_decoder metadata in investo_lora/tokenizer_config.json.


('investo_lora/tokenizer_config.json',
 'investo_lora/chat_template.jinja',
 'investo_lora/tokenizer.json')

In [ ]:
!ls -lh investo_lora

total 110M
-rw-r--r-- 1 root root 1.3K Jun 20 12:57 adapter_config.json
-rw------- 1 root root  93M Jun 20 12:57 adapter_model.safetensors
-rw-r--r-- 1 root root 3.8K Jun 20 12:57 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Jun 20 12:57 README.md
-rw-r--r-- 1 root root  50K Jun 20 12:57 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jun 20 12:57 tokenizer.json


In [ ]:
from google.colab import files
import shutil

shutil.make_archive(
    "investo_lora",
    "zip",
    "investo_lora"
)

files.download("investo_lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>